# Export and Import Workflow Manager Items Asynchronously #

This sample will export an existing Workflow Manager Item, create a new workflow item, then import the configuration to the newly created item in the same GIS. All this work will be done asynchronously.

### Make connections ###

In [ ]:
import arcgis
from arcgis.gis.workflowmanager import WorkflowManager, WorkflowManagerAdmin
import time

gis = arcgis.gis.GIS(url='https://hostname/portal', profile='my_profile')
wm_admin = WorkflowManagerAdmin(gis)
print("Created connection to Workflow Manager")

### Export existing item asynchronously ###

In [ ]:
# Export Item Asynchronously

item = gis.content.search('title:"Python Sample"')[0]
export_execution = wm_admin.export_item(item, run_async=True, download_location='C:\\Users\\exampleUser\\Desktop\\')

# Example: Can do additional computation while execution is running and before prompting for result
wm = WorkflowManager(item)
diagrams = wm.diagrams

# Now process the export_execution - result() blocks execution until the asynchronous work is finished and returns the last message received.
result = export_execution.result()

wmc_filepath = export_execution.export_location
mapping_file_path = export_execution.export_mapping_location

print(f'Result = {result}\n')
print(f'Here is the Exported ID: {export_execution.export_id}')
print(f'Here is the Exported WMC file location: {wmc_filepath}\n')
print(f'Here is the Exported WMC mapping file location: {mapping_file_path}\n')



### Create an Item and Import Configuration Asynchronously With a Mapping File

In [ ]:
#create a new item using wm_admin
new_item_id = wm_admin.create_item(name='New Workflow Item')
new_item = gis.content.get(new_item_id)

import_execution = wm_admin.import_item(new_item, wmc_filepath, run_async=True, overwrite_configuration=False, import_mapping_file=mapping_file_path)

while not import_execution.done():
    print(f'Progress = {import_execution.status}')
    print(f'{import_execution.messages}\n')
    time.sleep(5)

print(f'Status = {import_execution.status} \n')
print(f'Time elapsed {import_execution.elapse_time}')
print(f'Messages received: \n')
for m in import_execution.messages:
    print(f'{m.message} \n')

# Result() returns the last message received. This will inform you of the final state from importing
print(f'Result = {import_execution.result()}\n')
